In [1]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [3]:
# 1. Update pip to avoid dependency resolver issues
!python -m pip install --upgrade pip

# 2. Install all required packages (using standard hyphenated naming)
!pip install -q -U langchain langchain-community langchain-openai langchain-chroma chromadb tiktoken pypdf pysqlite3-binary

# 3. SQLite3 fix for ChromaDB compatibility (handles older environments automatically)
try:
    __import__("pysqlite3")
    import sys
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
except ImportError:
    pass

# 4. Verification imports
import chromadb
import langchain
import langchain_chroma
import langchain_community
import langchain_openai
import openai
import pypdf
import tiktoken

print("All packages successfully installed and imported.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


/tmp/ipykernel_960/2768840608.py:19: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


All packages successfully installed and imported.


In [6]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

doc1 = Document(
    page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
    metadata={"team": "Royal Challengers Bangalore"}
)
doc2 = Document(
    page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
    metadata={"team": "Mumbai Indians"}
)
doc3 = Document(
    page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
    metadata={"team": "Chennai Super Kings"}
)
doc4 = Document(
    page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
    metadata={"team": "Mumbai Indians"}
)
doc5 = Document(
    page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
    metadata={"team": "Chennai Super Kings"}
)

docs = [doc1, doc2, doc3, doc4, doc5]

In [7]:
vector_store = Chroma(
    embedding_function=OpenAIEmbeddings(),
    persist_directory='my_chroma_db',
    collection_name='sample'
)

In [11]:
# add documents
vector_store.add_documents(docs)

['6b82b237-22b8-448b-9620-f8a5f9bdc656',
 '506003ff-3ac3-450c-8ab4-7573a188e82a',
 'de0111c6-a402-4ca1-894c-8f9e12cab904',
 '63552bff-92f1-4dad-b33a-82c10a6d0483',
 '65c6957e-caca-4dfb-8dba-c5c6c9eed81e']

In [12]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['8fbf1480-5b96-4cd3-9825-43b5048c3d51',
  '1c00af2a-a7b0-45bb-bdb3-7a5d4fe94c27',
  '851bbc2d-74ac-4487-8670-73d0834ec370',
  '488712c0-e431-4309-baac-dbbee82fe431',
  'dcc635e2-12f6-4046-ab9b-a23d05add273',
  '6b82b237-22b8-448b-9620-f8a5f9bdc656',
  '506003ff-3ac3-450c-8ab4-7573a188e82a',
  'de0111c6-a402-4ca1-894c-8f9e12cab904',
  '63552bff-92f1-4dad-b33a-82c10a6d0483',
  '65c6957e-caca-4dfb-8dba-c5c6c9eed81e'],
 'embeddings': array([[-0.00210453, -0.00214285,  0.0268    , ..., -0.01707893,
         -0.00366616,  0.01357884],
        [-0.00268021, -0.00010323,  0.02815653, ..., -0.01501936,
          0.00590092, -0.01164922],
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        ...,
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        [-0.02714536,  0.00885395,  0.02699314, ..., -0.02592762,
          0.00900617, -0.01999116],
        [-0.01810451,  0.01281202, 

In [13]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='488712c0-e431-4309-baac-dbbee82fe431', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='63552bff-92f1-4dad-b33a-82c10a6d0483', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [15]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=1
)

[(Document(id='488712c0-e431-4309-baac-dbbee82fe431', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.35445845127105713)]

In [17]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='851bbc2d-74ac-4487-8670-73d0834ec370', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.6488258242607117),
 (Document(id='de0111c6-a402-4ca1-894c-8f9e12cab904', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.6488258242607117),
 (Document(id='dcc635e2-12f6-4046-ab9b-a23d05add273', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.6566494703292847),
 (Document(id='65c6957e-caca-4dfb-8dba-c5c6c9eed81e', metadata={'team': 'Chennai S

In [18]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)


In [19]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['8fbf1480-5b96-4cd3-9825-43b5048c3d51',
  '1c00af2a-a7b0-45bb-bdb3-7a5d4fe94c27',
  '851bbc2d-74ac-4487-8670-73d0834ec370',
  '488712c0-e431-4309-baac-dbbee82fe431',
  'dcc635e2-12f6-4046-ab9b-a23d05add273',
  '6b82b237-22b8-448b-9620-f8a5f9bdc656',
  '506003ff-3ac3-450c-8ab4-7573a188e82a',
  'de0111c6-a402-4ca1-894c-8f9e12cab904',
  '63552bff-92f1-4dad-b33a-82c10a6d0483',
  '65c6957e-caca-4dfb-8dba-c5c6c9eed81e'],
 'embeddings': array([[-0.00210453, -0.00214285,  0.0268    , ..., -0.01707893,
         -0.00366616,  0.01357884],
        [-0.00268021, -0.00010323,  0.02815653, ..., -0.01501936,
          0.00590092, -0.01164922],
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        ...,
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        [-0.02714536,  0.00885395,  0.02699314, ..., -0.02592762,
          0.00900617, -0.01999116],
        [-0.01810451,  0.01281202, 

In [20]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [22]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['8fbf1480-5b96-4cd3-9825-43b5048c3d51',
  '1c00af2a-a7b0-45bb-bdb3-7a5d4fe94c27',
  '851bbc2d-74ac-4487-8670-73d0834ec370',
  '488712c0-e431-4309-baac-dbbee82fe431',
  'dcc635e2-12f6-4046-ab9b-a23d05add273',
  '6b82b237-22b8-448b-9620-f8a5f9bdc656',
  '506003ff-3ac3-450c-8ab4-7573a188e82a',
  'de0111c6-a402-4ca1-894c-8f9e12cab904',
  '63552bff-92f1-4dad-b33a-82c10a6d0483',
  '65c6957e-caca-4dfb-8dba-c5c6c9eed81e'],
 'embeddings': array([[-0.00210453, -0.00214285,  0.0268    , ..., -0.01707893,
         -0.00366616,  0.01357884],
        [-0.00268021, -0.00010323,  0.02815653, ..., -0.01501936,
          0.00590092, -0.01164922],
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        ...,
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        [-0.02714536,  0.00885395,  0.02699314, ..., -0.02592762,
          0.00900617, -0.01999116],
        [-0.01810451,  0.01281202, 